In [30]:
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.ensemble import IsolationForest
import pandas as pd 
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.cluster import KMeans
Raw_Path="ecommerce_transactions.csv.csv"
Cleaned_Path="ecommerce_transactions_cleaned.csv.csv"
FLAGGED_PATH = "ecommerce_transactions_anomalies.csv.csv"

CONTAMINATION=0.02
Random_state=42
N_CLUSTERS=6



In [31]:
# step 1: cleaning
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    n_start = len(df)
    text_cols = ["User_Name", "Country", "Product_Category", "Payment_Method"]
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
        df["Country"] = df["Country"].str.title().replace({"Uk": "UK", "Usa": "USA"})
        df["Product_Category"] = df["Product_Category"].str.title()
        df["Payment_Method"] = df["Payment_Method"].str.title().replace({"Upi": "UPI", "Paypal": "PayPal"})

    df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], errors="coerce")
    df = df.dropna(subset=["Transaction_Date"])

    df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
    df["Purchase_Amount"] = pd.to_numeric(df["Purchase_Amount"], errors="coerce")
    df = df.dropna(subset=["Age", "Purchase_Amount"])
    df = df[(df["Age"] >= 10) & (df["Age"] <= 100)]
    df = df[df["Purchase_Amount"] > 0]

    df = df.drop_duplicates()
    df = df.drop_duplicates(subset=["Transaction_ID"], keep="first")
    df = df.dropna(subset=["Transaction_ID", "User_Name", "Country", "Product_Category", "Payment_Method"])

    df["Age"] = df["Age"].astype(int)
    df = df.sort_values("Transaction_ID").reset_index(drop=True)
    print(f"cleaning:{n_start}->{len(df)} rows({n_start-len(df)} removed)")
    return df

In [20]:
#Feature Engineering 
def build_features(df:pd.DataFrame)->pd.DataFrame:
    feat=pd.DataFrame(index=df.index)
    feat["purchase_amount"]=df["Purchase_Amount"]
    feat["age"]=df["Age"]
    feat["day_of_week"]=df["Transaction_Date"].dt.dayofweek
    feat["month"]=df["Transaction_Date"].dt.month
    feat["is_weekend"]=(feat["day_of_week"]>=5).astype(int)
    grp_cat=df.groupby("Product_Category")["Purchase_Amount"]
    cat_mean=grp_cat.transform("mean")
    cat_std=grp_cat.transform("std").replace(0,np.nan)
    feat["user_amount_zscore"]=((df["Purchase_Amount"]-cat_mean)/cat_std).fillna(0)
    feat["user_payment_diversity"]=df.groupby("User_Name")["Payment_Method"].transform("nunique")
    feat["user_country_diversity"]=df.groupby("User_Name")["Country"].transform("nunique")
    feat["user_total_transactions"]=df.groupby("User_Name")["Transaction_ID"].transform("count")
    year_month=df["Transaction_Date"].dt.to_period("M")
    feat["user_transacations_this_month"]=df.groupby(
        ["User_Name", year_month]
    ) ["Transaction_ID"].transform("count")
    cat_cols=["Country", "Product_Category", "Payment_Method"]
    encoder=OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoded=encoder.fit_transform(df[cat_cols])
    encoded_df=pd.DataFrame(
        encoded,columns=encoder.get_feature_names_out(cat_cols),index=df.index
    )

    feat= pd.concat([feat,encoded_df], axis=1)
    return feat


In [21]:
def run_isolation_forest(features:pd.DataFrame, contamination:float)->pd.DataFrame:
    model=IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=Random_state,
        n_jobs=-1
    )
    model.fit(features)

    scores=model.decision_function(features)
    preds=model.predict(features)
    result=pd.DataFrame(index=features.index)
    result["anomaly_score"]=scores
    result["is_anomaly"]=(preds==-1)
    return result

def main():
    raw=pd.read_csv(Raw_Path)
    clean=clean_data(raw)
    clean.to_csv(Cleaned_Path,index=False)
    print(f"Saved cleaned data->{Cleaned_Path}")
    features=build_features(clean)
    result=run_isolation_forest(features,contamination)
    flagged=clean.join(result)
    flagged=flagged.sort_values("anomaly_score")
    flagged.to_csv(FLAGGED_PATH,index=False)
    print(f"Saved flagged data->{FLAGGED_PATH}")

    n_anom=flagged["is_anomaly"].sum()
    print(f"/nFLAGGED{n_anom}anomalies out of {len(flagged)}"
          f"({n_anom/len(flagged):.2%})")

if __name__=="__main__":
    main()



Raw_Path="ecommerce_transactions.csv.csv"

result_check = pd.read_csv(FLAGGED_PATH).sort_values("anomaly_score")
result_check.head(20)

cleaning:50000->50000 rows(0 removed)
Saved cleaned data->ecommerce_transactions_cleaned.csv.csv
Saved flagged data->ecommerce_transactions_anomalies.csv.csv
/nFLAGGED1000anomalies out of 50000(2.00%)


,Transaction_ID,User_Name,Age,Country,Product_Category,Purchase_Amount,Payment_Method,Transaction_Date,anomaly_score,is_anomaly
0,6026,Isabella Thompson,67,Mexico,Sports,101.74,UPI,2025-03-01,-0.030543,True
1,15562,Elijah Hall,18,USA,Beauty,213.36,Credit Card,2024-12-01,-0.025370,True
2,35701,Oliver Rodriguez,19,Mexico,Home & Kitchen,838.10,Credit Card,2024-12-08,-0.024199,True
3,24054,Noah White,69,India,Electronics,989.14,Cash On Delivery,2024-01-27,-0.023137,True
4,30456,Elijah Thompson,20,USA,Books,18.87,Credit Card,2025-03-02,-0.021997,True
5,24184,James Lewis,68,USA,Beauty,980.24,UPI,2024-12-01,-0.020464,True
6,11398,Oliver Rodriguez,60,USA,Toys,78.22,PayPal,2023-11-26,-0.020102,True
7,7226,Elijah White,23,USA,Beauty,22.36,Cash On Delivery,2024-11-03,-0.019839,True
8,8634,Emma Clark,65,UK,Beauty,995.58,Credit Card,2024-03-17,-0.019750,True
9,36819,Elijah Allen,63,Mexico,Electronics,980.73,Cash On Delivery,2023-07-02,-0.019649,True


In [22]:
result_check["Product_Category"].value_counts()

Product_Category
Toys              6392
Electronics       6320
Sports            6312
Books             6253
Clothing          6224
Grocery           6215
Home & Kitchen    6209
Beauty            6075
Name: count, dtype: int64

In [23]:

anomalies_only = result_check[result_check["is_anomaly"]]
anomalies_only["Product_Category"].value_counts()


Product_Category
Beauty            225
Electronics       194
Books             192
Sports            165
Toys               84
Grocery            53
Clothing           44
Home & Kitchen     43
Name: count, dtype: int64

In [33]:
#Anomaly Detection-k means 
def run_kmeans(features:pd.DataFrame,n_clusters:int,contamination:float)->pd.DataFrame:
    #k-means is distance based , so features must be on the same scale first 
    scaler=StandardScaler()
    scaled=scaler.fit_transform(features)

    model=KMeans(n_clusters=n_clusters, random_state=Random_state, n_init=10)
    cluster_labels=model.fit_predict(scaled)

    #distance from each point to its own clusters centroid 
    centroids=model.cluster_centers_
    distances=np.linalg.norm(scaled-centroids[cluster_labels],axis=1)

    #flag the furthest 'contamination' as anomalies 
    threshold=np.quantile(distances, 1-contamination)

    result=pd.DataFrame(index=features.index)
    result["cluster"]=cluster_labels
    result["distance_to_centroid"]=distances
    result["kmeans_is_anomaly"]=distances>=threshold
    return result

def main():
    raw=pd.read_csv(Raw_Path)
    clean=clean_data(raw)
    clean.to_csv(Cleaned_Path,index=False)
    print(f"Saved cleaned data->{Cleaned_Path}")

    features=build_features(clean)

    iso_result=run_isolation_forest(features, CONTAMINATION)
    kmeans_result=run_kmeans(features, N_CLUSTERS, CONTAMINATION)
    flagged=clean.join(iso_result).join (kmeans_result)
    flagged["flagged_by_both"]=flagged["is_anomaly"]& flagged["kmeans_is_anomaly"]
    flagged=flagged.sort_values("anomaly_score")
    flagged.to_csv(FLAGGED_PATH,index=False)
    print(f"Saved flagged data->{FLAGGED_PATH}")

    n_iso=flagged["is_anomaly"].sum()
    n_kmeans=flagged["kmeans_is_anomaly"].sum()
    n_both=flagged["flagged_by_both"].sum()

    print(f"/nIsolation Forest flagged:{n_iso}({n_iso/len(flagged):.2%})")
    print(f"K-means flagged:          {n_kmeans} ({n_kmeans/len(flagged):.2%})")
    print(f"Flagged by both:           {n_both} ({n_both/len(flagged):.2%})")


if __name__=="__main__":
    main()

cleaning:50000->50000 rows(0 removed)
Saved cleaned data->ecommerce_transactions_cleaned.csv.csv
Saved flagged data->ecommerce_transactions_anomalies.csv.csv
/nIsolation Forest flagged:1000(2.00%)
K-means flagged:          1000 (2.00%)
Flagged by both:           281 (0.56%)


In [35]:
result_check = pd.read_csv(FLAGGED_PATH)
result_check.sort_values("distance_to_centroid", ascending=False).head(20)

result_check[result_check["flagged_by_both"]].sort_values("anomaly_score")

only_iso = result_check[result_check["is_anomaly"] & ~result_check["kmeans_is_anomaly"]]
only_kmeans = result_check[~result_check["is_anomaly"] & result_check["kmeans_is_anomaly"]]